# scPoli校正流程 Python版

把`原始基因表达h5ad`和`cell×10 scPoli h5ad`合并，按Seurat代码逻辑做epithelial subset、Normalize、UMAP过滤、scPoli聚类、local purity过滤、marker DotPlot和统计输出。

In [1]:
import os,numpy as np,pandas as pd,scanpy as sc,scipy.sparse as sp,matplotlib.pyplot as plt
import os
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
sc.settings.verbosity=2
sc.settings.seed=1234
np.random.seed(1234)

In [2]:
raw_h5ad="../../7_mapping_new_data/0511_rename_noIAISR/output_human/Atlas_allhuman_with_measure_allgeneID-inner-counts-annot.h5ad"##所有人类数据 原始counts + annot,现在这个文件中的cell_type_level1是ground truth
scpoli_h5ad="../../7_mapping_new_data/0507_no_IAISR/output_human/level1_human_atlas_nogene_noIAISR_2.h5ad"### all human scpoli data without gene names, but with annotation
outdir="./output_allhuman"
counts_layer="rounded_corrected_counts"
original_celltype_col="cell_type_level1"##原始文件里的标签
os.makedirs(outdir,exist_ok=True)

In [3]:
#####读取文件 按 adata_scpoli 的细胞ID 对齐 adata_raw
adata_raw=sc.read_h5ad(raw_h5ad)
adata_scpoli=sc.read_h5ad(scpoli_h5ad)
print(adata_raw)
print(adata_raw.obs_names)
print(adata_scpoli)
print(adata_scpoli.obs_names)

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


AnnData object with n_obs × n_vars = 1019138 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species'
    var: 'original_gene_names', 'ensembl_id'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
Index(['AAACCCAAGAGGTTAT-1', 'AAACCCAAGCAACAAT-1', 'AAACCCAAGGAGTCTG-1',
       'AAACCCACAACTCATG-1', 'AAACCCACAGCTCTGG-1', 'AAACCCAGTCAAGGCA-1',
       'AAACCCAGTCAATCTG-1', 'AAACCCAGTCACCCTT-1', 'AAACCCAGTGCATTAC-1',
       'AAACCCAGTGCGGCTT-1',
       ...
       'ATAACGCAGTACGCCC_H_plaque', 'CACATAGGTGTCTGAT_H_plaque',
       'CCATTCGAGTAGCCGA_H_plaque', 'GAATAAGAGAAAGTGG_H_plaque',
       'GACCAATCATGGTAGG_H_plaque', 'GACCTGGGTCT

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/anndata/_core/anndata.py:1897: UserWarning: Observation names are not unique. To make them unique, call `.obs_names_make_unique`.
  utils.warn_names_duplicates("obs")


In [ ]:
###得到共同的细胞
adata_raw.obs_names=adata_raw.obs_names.astype(str)
adata_scpoli.obs_names=adata_scpoli.obs_names.astype(str)
##保留原始barcode
adata_raw.obs["barcode_for_match"]=adata_raw.obs_names
adata_scpoli.obs["barcode_for_match"]=adata_scpoli.obs_names.str[:-2]
##用 sample + barcode 构建唯一ID
adata_raw.obs["match_id"]=adata_raw.obs["sample"].astype(str)+"__"+adata_raw.obs["barcode_for_match"].astype(str)
adata_scpoli.obs["match_id"]=adata_scpoli.obs["sample"].astype(str)+"__"+adata_scpoli.obs["barcode_for_match"].astype(str)
##如果sample仍不唯一，就改成 dataset + sample + barcode
if adata_raw.obs["match_id"].duplicated().sum()>0 or adata_scpoli.obs["match_id"].duplicated().sum()>0:
    adata_raw.obs["match_id"]=adata_raw.obs["dataset"].astype(str)+"__"+adata_raw.obs["sample"].astype(str)+"__"+adata_raw.obs["barcode_for_match"].astype(str)
    adata_scpoli.obs["match_id"]=adata_scpoli.obs["dataset"].astype(str)+"__"+adata_scpoli.obs["sample"].astype(str)+"__"+adata_scpoli.obs["barcode_for_match"].astype(str)
print("raw duplicated match_id:",adata_raw.obs["match_id"].duplicated().sum())
print("scpoli duplicated match_id:",adata_scpoli.obs["match_id"].duplicated().sum())
if adata_raw.obs["match_id"].duplicated().sum()>0:
    raise ValueError("adata_raw match_id仍有重复，不能安全匹配")
if adata_scpoli.obs["match_id"].duplicated().sum()>0:
    raise ValueError("adata_scpoli match_id仍有重复，不能安全匹配")
adata_raw.obs_names=adata_raw.obs["match_id"].astype(str)
adata_scpoli.obs_names=adata_scpoli.obs["match_id"].astype(str)

##空值填充为query
if "atlas_key" not in adata_scpoli.obs:
    adata_scpoli.obs["atlas_key"]="query"
else:
    adata_scpoli.obs["atlas_key"]=adata_scpoli.obs["atlas_key"].astype("object")
    adata_scpoli.obs.loc[adata_scpoli.obs["atlas_key"].isna(),"atlas_key"]="query"
    adata_scpoli.obs.loc[adata_scpoli.obs["atlas_key"].astype(str).isin(["","nan","None","NA"]),"atlas_key"]="query"

print(adata_scpoli.obs["atlas_key"].value_counts(dropna=False))
##以 scpoli 为准，从 raw 中取相同细胞
scp_ids=adata_scpoli.obs_names
missing=scp_ids[~scp_ids.isin(adata_raw.obs_names)]
print("scpoli cells:",len(scp_ids))
print("missing in raw:",len(missing))
if len(missing)>0:
    print(missing[:50].tolist())
    raise ValueError("有 scpoli 细胞在 raw 中找不到")
adata=adata_raw[scp_ids,:].copy()
adata_com_scpoli=adata_scpoli.copy()
assert np.all(adata.obs_names==adata_com_scpoli.obs_names)
print("aligned adata:",adata)
print("aligned adata_com_scpoli:",adata_com_scpoli)

raw duplicated match_id: 0
scpoli duplicated match_id: 0
atlas_key
query    548208
ref      470566
Name: count, dtype: int64
scpoli cells: 1018774
missing in raw: 0
aligned adata: AnnData object with n_obs × n_vars = 1018774 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id'
    var: 'original_gene_names', 'ensembl_id'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts'
aligned adata_com_scpoli: AnnData object with n_obs × n_vars = 1018774 × 10
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_conta

In [5]:
adata_query = adata[adata.obs['atlas_key'] == 'query']
adata_query.obs['cell_type_level1'].value_counts()

cell_type_level1
T cell                   183446
B cell                    56046
Macrophage                55696
Natural killer cell       51294
Fibroblast                49419
Smooth muscle cell        42831
Endothelial cell          30615
Dendritic cell            24663
Monocyte                  16448
Neutrophil                14157
Pericyte                   9864
Basophil                   6162
Mast cell                  4427
Erythrocyte/Erythroid      3140
Name: count, dtype: int64

In [6]:
adata_query = adata[adata.obs['atlas_key'] == 'ref']
adata_query.obs['cell_type_level1'].value_counts()

cell_type_level1
T cell                   154319
Neutrophil                68634
Macrophage                48793
Fibroblast                39718
Endothelial cell          34734
B cell                    34289
Smooth muscle cell        31685
Natural killer cell       20001
Monocyte                  16440
Dendritic cell             9879
Pericyte                   9121
Mast cell                  2532
Erythrocyte/Erythroid       302
Basophil                    119
Name: count, dtype: int64

In [7]:
adata.obs['atlas_key'].value_counts()

atlas_key
query    548208
ref      470566
Name: count, dtype: int64

In [8]:
adata_com_scpoli.write("./output_allhuman/allhuman_scpoli_com.h5ad")
adata.write("./output_allhuman/allhuman_raw_com.h5ad")

In [26]:
adata_com_scpoli = sc.read_h5ad("./output_allhuman/allhuman_scpoli_com.h5ad")
adata = sc.read_h5ad("./output_allhuman/allhuman_raw_com.h5ad")

In [27]:
##选择原始 counts 矩阵，并检查它像不像真正的原始计数
if counts_layer in adata.layers:
    adata.X=adata.layers[counts_layer].copy()
    print("use layer:",counts_layer)
else:
    print("layer not found,use adata.X:",counts_layer)
adata.layers["counts"]=adata.X.copy()
x=adata.layers["counts"]
v=x.data if sp.issparse(x) else np.ravel(x)
v=v[:min(len(v),1000000)]
print("counts min",np.min(v),"max",np.max(v),"integer-like",np.allclose(v,np.round(v)))

use layer: rounded_corrected_counts
counts min 1 max 5134 integer-like True


In [28]:
adata_com_scpoli.obs['cell_type_pred'].value_counts()

cell_type_pred
T cell                   183446
B cell                    56046
Macrophage                55696
Natural killer cell       51294
Fibroblast                49419
Smooth muscle cell        42831
Endothelial cell          30615
Dendritic cell            24663
Monocyte                  16448
Neutrophil                14157
Pericyte                   9864
Basophil                   6162
Mast cell                  4427
Erythrocyte/Erythroid      3140
Name: count, dtype: int64

In [29]:
adata_com_scpoli.obs['cell_type_level1'].value_counts()

cell_type_level1
unknown                  548208
T cell                   154319
Neutrophil                68634
Macrophage                48793
Fibroblast                39718
Endothelial cell          34734
B cell                    34289
Smooth muscle cell        31685
Natural killer cell       20001
Monocyte                  16440
Dendritic cell             9879
Pericyte                   9121
Mast cell                  2532
Erythrocyte/Erythroid       302
Basophil                    119
Name: count, dtype: int64

In [30]:
adata.obs['cell_type_level1'].value_counts()

cell_type_level1
T cell                   337765
Macrophage               104489
B cell                    90335
Fibroblast                89137
Neutrophil                82791
Smooth muscle cell        74516
Natural killer cell       71295
Endothelial cell          65349
Dendritic cell            34542
Monocyte                  32888
Pericyte                  18985
Mast cell                  6959
Basophil                   6281
Erythrocyte/Erythroid      3442
Name: count, dtype: int64

In [32]:
#####添加需要的obs
run_tag="precorrect"
for c in ["leiden","cell_type_pred","cell_type_uncert","query","cell_type_pred_ref","cell_type_level1_human","atlas_key"]:
    if c in adata_com_scpoli.obs:
        adata.obs[c]=adata_com_scpoli.obs[c].values
        print("+obs",c)
X=adata_com_scpoli.X.toarray() if sp.issparse(adata_com_scpoli.X) else np.asarray(adata_com_scpoli.X)
adata.obsm["X_scPoli"]=X
if "X_umap" in adata_com_scpoli.obsm:
    adata.obsm["X_umap"]=np.asarray(adata_com_scpoli.obsm["X_umap"])
print("X_scPoli",adata.obsm["X_scPoli"].shape)
print("X_umap",adata.obsm["X_umap"].shape if "X_umap" in adata.obsm else None)
adata.write(os.path.join(outdir,f"allhuman_raw_counts_scpoli_{run_tag}.h5ad"))

+obs leiden
+obs cell_type_pred
+obs cell_type_uncert
+obs query
+obs cell_type_pred_ref
+obs cell_type_level1_human
+obs atlas_key
X_scPoli (1018774, 10)
X_umap (1018774, 2)


In [33]:
##合并后先画一张全体细胞的 UMAP，检查 cell_type_pred 是否正常
if "X_umap" in adata.obsm and "cell_type_pred" in adata.obs:
    sc.pl.umap(adata,color="cell_type_pred",legend_loc="on data",frameon=False,size=1,show=False)
    plt.savefig(os.path.join(outdir,"umap_cell_type_pred_full.pdf"),bbox_inches="tight")
    plt.close()

/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:1251: FutureWarning: The default value of 'ignore' for the `na_action` parameter in pandas.Categorical.map is deprecated and will be changed to 'None' in a future version. Please set na_action to the desired value to avoid seeing this warning
  color_vector = pd.Categorical(values.map(color_map))
/home/lixiangyu/anaconda3/envs/zr_r420/lib/python3.9/site-packages/scanpy/plotting/_tools/scatterplots.py:394: UserWarning: No data for colormapping provided via 'c'. Parameters 'cmap' will be ignored
  cax = scatter(


In [34]:
adata

AnnData object with n_obs × n_vars = 1018774 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_pred_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts', 'counts'

In [36]:
adata.obs['cell_type_level1'].value_counts()

cell_type_level1
T cell                   337765
Macrophage               104489
B cell                    90335
Fibroblast                89137
Neutrophil                82791
Smooth muscle cell        74516
Natural killer cell       71295
Endothelial cell          65349
Dendritic cell            34542
Monocyte                  32888
Pericyte                  18985
Mast cell                  6959
Basophil                   6281
Erythrocyte/Erythroid      3442
Name: count, dtype: int64

In [37]:
adata.obs['cell_type_pred'].value_counts()

cell_type_pred
T cell                   183446
B cell                    56046
Macrophage                55696
Natural killer cell       51294
Fibroblast                49419
Smooth muscle cell        42831
Endothelial cell          30615
Dendritic cell            24663
Monocyte                  16448
Neutrophil                14157
Pericyte                   9864
Basophil                   6162
Mast cell                  4427
Erythrocyte/Erythroid      3140
Name: count, dtype: int64

### mac/mono/dc/neu

In [38]:
##只保留要校正的细胞类型--这里根据cell_type_pred来选
mac_mono_dc_neu_cells=["Macrophage",'Monocyte','Dendritic cell','Neutrophil']
if "cell_type_pred" not in adata.obs:raise ValueError("cell_type_pred not found")
mac_mono_dc_neu=adata[adata.obs["cell_type_pred"].astype(str).isin(mac_mono_dc_neu_cells),:].copy()
print(mac_mono_dc_neu)
print(mac_mono_dc_neu.obs["cell_type_pred"].value_counts())

AnnData object with n_obs × n_vars = 110964 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_pred_colors'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts', 'counts'
cell_type_pred
Macrophage        55696
Dendritic cell    24663
Monocyte          16448
Neutrophil        14157
Name: count, dtype: int64


In [20]:
###对选取的数据归一化
mac_mono_dc_neu.X=mac_mono_dc_neu.layers["counts"].copy()
mac_mono_dc_neu.layers["counts"]=mac_mono_dc_neu.X.copy()
sc.pp.normalize_total(mac_mono_dc_neu,target_sum=1e4)
sc.pp.log1p(mac_mono_dc_neu)
mac_mono_dc_neu.layers["lognorm"]=mac_mono_dc_neu.X.copy()
x=mac_mono_dc_neu.layers["lognorm"]
v=x.data if sp.issparse(x) else np.ravel(x)
v=v[:min(len(v),1000000)]
print("lognorm min",np.min(v),"max",np.max(v),"integer-like should be False",np.allclose(v,np.round(v)))

normalizing counts per cell
    finished (0:00:15)
lognorm min 0.061395705 max 7.7495246 integer-like should be False False


In [21]:
mac_mono_dc_neu

AnnData object with n_obs × n_vars = 110964 × 28868
    obs: 'sample', 'dataset', 'symptoms', 'scDblFinder.class', 'scDblFinder.score', 'scDblFinder.weighted', 'scDblFinder.cxds_score', 'decontX_contamination', 'decontX_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'n_counts', 'cell_type_level1', 'scDblFinder.sample', 'gender', 'age', 'intervention', 'tissue', 'atlas_key', 'species', 'barcode_for_match', 'match_id', 'leiden', 'cell_type_pred', 'cell_type_uncert', 'query', 'cell_type_pred_ref', 'cell_type_level1_human'
    var: 'original_gene_names', 'ensembl_id'
    uns: 'cell_type_pred_colors', 'log1p'
    obsm: 'X_scPoli', 'X_umap'
    layers: 'raw_decontXcounts', 'rounded_corrected_counts', 'uncorrected_counts', 'counts', 'lognorm'

In [22]:
mac_mono_dc_neu.obs['cell_type_pred'].value_counts()

cell_type_pred
Macrophage        55696
Dendritic cell    24663
Monocyte          16448
Neutrophil        14157
Name: count, dtype: int64

In [ ]:
mac_mono_dc_neu.obs['cell_type_level1'].value_counts()

cell_type_level1
Macrophage        55696
Dendritic cell    24663
Monocyte          16448
Neutrophil        14157
Name: count, dtype: int64

In [24]:
adata.obs['cell_type_level1'].value_counts()

cell_type_level1
T cell                   337765
Macrophage               104489
B cell                    90335
Fibroblast                89137
Neutrophil                82791
Smooth muscle cell        74516
Natural killer cell       71295
Endothelial cell          65349
Dendritic cell            34542
Monocyte                  32888
Pericyte                  18985
Mast cell                  6959
Basophil                   6281
Erythrocyte/Erythroid      3442
Name: count, dtype: int64

In [25]:
adata.obs['cell_type_level1'].value_counts()

cell_type_level1
T cell                   337765
Macrophage               104489
B cell                    90335
Fibroblast                89137
Neutrophil                82791
Smooth muscle cell        74516
Natural killer cell       71295
Endothelial cell          65349
Dendritic cell            34542
Monocyte                  32888
Pericyte                  18985
Mast cell                  6959
Basophil                   6281
Erythrocyte/Erythroid      3442
Name: count, dtype: int64

In [24]:
remove_types=[ 
    "T cell",
    "Fibroblast",
    "B cell",
    "Smooth muscle cell",
    "Endothelial cell",
    "Natural killer cell",
    "Pericyte",
    "Mast cell",
    "Basophil",
    "Erythrocyte/Erythroid"]
orig_col=None
for c in [original_celltype_col,"cell_type_level1","cell_type_level1_human","cell_type_pred_ref"]:
    if c in mac_mono_dc_neu.obs:
        orig_col=c;break
if orig_col is not None:
    n=mac_mono_dc_neu.n_obs
    mac_mono_dc_neu=mac_mono_dc_neu[~mac_mono_dc_neu.obs[orig_col].astype(str).isin(remove_types),:].copy()
    print("use",orig_col,"removed",n-mac_mono_dc_neu.n_obs)
    print(mac_mono_dc_neu.obs[orig_col].value_counts())
else:
    print("skip contaminant removal:no original celltype column")

use cell_type_level1 removed 4
cell_type_level1
Macrophage        55692
Dendritic cell    24663
Monocyte          16447
Neutrophil        14158
Name: count, dtype: int64


In [ ]:
if "X_umap" not in epi.obsm:raise ValueError("X_umap not found")
n=epi.n_obs
keep=~(epi.obsm["X_umap"][:,1]<3)
epi=epi[keep,:].copy()
print("keep UMAP_2>=3","before",n,"after",epi.n_obs,"removed",n-epi.n_obs)
sc.pl.umap(epi,color="cell_type_pred",legend_loc="on data",frameon=False,size=2,show=False)
plt.savefig(os.path.join(outdir,"umap_epi_after_coord_filter_cell_type_pred.pdf"),bbox_inches="tight")
plt.close()
if "dataset" in epi.obs:
    sc.pl.umap(epi,color="dataset",frameon=False,size=2,show=False)
    plt.savefig(os.path.join(outdir,"umap_epi_after_coord_filter_dataset.pdf"),bbox_inches="tight")
    plt.close()

In [ ]:
sc.pp.neighbors(epi,use_rep="X_scPoli",n_neighbors=150,metric="euclidean")
sc.tl.leiden(epi,resolution=3,key_added="leiden_scpoli_res3",random_state=1234)
sc.tl.umap(epi,min_dist=1.1,random_state=1234)
for c in ["leiden_scpoli_res3","cell_type_pred","dataset"]:
    if c in epi.obs:
        sc.pl.umap(epi,color=c,legend_loc="on data" if c!="dataset" else "right margin",frameon=False,size=2,show=False)
        plt.savefig(os.path.join(outdir,f"umap_epi_res3_scpoli_{c}.pdf"),bbox_inches="tight")
        plt.close()

In [ ]:
emb=epi.obsm["X_scPoli"]
labels=epi.obs["cell_type_pred"].astype(str).values
K=30
nn=NearestNeighbors(n_neighbors=K+1,metric="euclidean").fit(emb)
idx=nn.kneighbors(emb,return_distance=False)[:,1:]
purity=np.array([np.mean(labels[idx[i]]==labels[i]) for i in range(epi.n_obs)])
epi.obs["local_purity"]=purity
print(pd.Series(purity).describe())

In [ ]:
purity_thresh={"PMC":0.3,
               "Parietal cell":0.2,
               "Chief cell":0.1,
               "Enteroendocrine cell":0.1,
               "Gastric progenitor cell":0.15,
               "Neck-chief hybrid cell":0.1,
               "Neck cell":0.1,
               "Metaplastic stem cell":0.1}
ct=epi.obs["cell_type_pred"].astype(str).values
p=epi.obs["local_purity"].values
keep=np.array([p[i]>=purity_thresh.get(ct[i],-np.inf) for i in range(epi.n_obs)])
summary=pd.DataFrame({"celltype":ct,"keep":keep}).groupby("celltype").agg(before=("keep","size"),after=("keep","sum"))
summary["removed"]=summary["before"]-summary["after"]
summary["pct_removed"]=summary["removed"]/summary["before"]*100
summary["threshold"]=summary.index.map(purity_thresh)
summary=summary.reset_index()[["celltype","threshold","before","after","removed","pct_removed"]]
print(summary.to_string(index=False))
summary.to_csv(os.path.join(outdir,"local_purity_removed_summary.csv"),index=False)
print("Before",epi.n_obs,"After",int(keep.sum()),"Removed",int((~keep).sum()),f"({(~keep).sum()/epi.n_obs*100:.2f}%)")
epi_clean=epi[keep,:].copy()

In [ ]:
sc.pl.umap(epi_clean,color="cell_type_pred",legend_loc="on data",frameon=False,size=2,show=False)
plt.savefig(os.path.join(outdir,"epi_umap_cleaned_cell_type_pred.pdf"),bbox_inches="tight")
plt.savefig(os.path.join(outdir,"epi_umap_cleaned_cell_type_pred.png"),dpi=300,bbox_inches="tight")
plt.close()
for ct0 in epithelial_cells:
    col=f"highlight_{ct0}"
    epi_clean.obs[col]=np.where(epi_clean.obs["cell_type_pred"].astype(str)==ct0,ct0,"Other")
    sc.pl.umap(epi_clean,color=col,groups=[ct0],frameon=False,size=2,title=f"{ct0} (n={(epi_clean.obs['cell_type_pred'].astype(str)==ct0).sum()})",show=False)
    plt.savefig(os.path.join(outdir,f"epi_umap_cleaned_highlight_{ct0.replace(' ','_').replace('/','_')}.pdf"),bbox_inches="tight")
    plt.close()

In [ ]:
marker_epi={"Enteroendocrine":["RFX6","SCG5","SCGN","CHGA"],
            "Metaplastic/stem":["OLFM4","LGR5","SOX9"],
            "Parietal":["ATP4A","ATP4B","GIF"],
            "Cycling progenitor":["TOP2A","MKI67","CENPF","CDK1"],
            "PMC":["MUC5AC","TFF1"],
            "Chief-like":["CEACAM5","CEACAM6","CEACAM7"],
            "Neck-chief hybrid/SPEM":["PGA3","PGA4","PGA5","MUC6","TFF2","CD44","LGR5","ANXA10","BHLHA15","TNFRSF19","MKI67","TOP2A"]}
marker_epi={k:[g for g in v if g in epi_clean.var_names] for k,v in marker_epi.items()}
marker_epi={k:v for k,v in marker_epi.items() if len(v)>0}
if marker_epi:
    sc.pl.dotplot(epi_clean,var_names=marker_epi,groupby="cell_type_pred",standard_scale="var",show=False)
    plt.savefig(os.path.join(outdir,"dot_marker_transfer_epi.pdf"),bbox_inches="tight")
    plt.close()
else:
    print("no epithelial marker genes found")

In [ ]:
marker_contam={"T cell":["CD2","CD3D","CD3E","CD3G","TRBC2"],
               "Smooth muscle":["ACTA2","ACTG2","MYH11","SYNPO2","LMOD1"],
               "Plasma":["MZB1","CD79A","ENAM","IGHA2"],"B cell":["MS4A1","BANK1"],
               "Mast":["TPSAB1","TPSB2","CPA3","KIT"],
               "Macrophage":["CD14","CD163","CSF1R","C1QC"],
               "Fibroblast":["COL1A2","DCN","COL3A1","LUM","COL6A1","CXCL14"],
               "Endothelial":["PECAM1","VWF","ENG","PLVAP"]}
marker_contam={k:[g for g in v if g in epi_clean.var_names] for k,v in marker_contam.items()}
marker_contam={k:v for k,v in marker_contam.items() if len(v)>0}
if marker_contam:
    sc.pl.dotplot(epi_clean,var_names=marker_contam,groupby="cell_type_pred",standard_scale="var",show=False)
    plt.savefig(os.path.join(outdir,"dot_marker_transfer_contamination.pdf"),bbox_inches="tight")
    plt.close()
else:
    print("no contamination marker genes found")

In [ ]:
if "cell_type_uncert" in epi_clean.obs:
    u=pd.to_numeric(epi_clean.obs["cell_type_uncert"],errors="coerce")
    print(u.describe())
    print("SD",u.std(),"NA",u.isna().sum())
    q=u.quantile([0,0.05,0.1,0.25,0.5,0.75,0.9,0.95,0.99,1])
    print(q)
    u.describe().to_csv(os.path.join(outdir,"cell_type_uncert_summary.csv"))
    q.to_csv(os.path.join(outdir,"cell_type_uncert_quantiles.csv"))
    plt.figure(figsize=(6,4));plt.hist(u.dropna(),bins=50);plt.xlabel("Uncertainty");plt.ylabel("Cell count");plt.title("Distribution of cell_type_uncert");plt.tight_layout()
    plt.savefig(os.path.join(outdir,"cell_type_uncert_hist.pdf"));plt.close()
else:
    print("cell_type_uncert not found")

In [ ]:
summary_long=epi_clean.obs.assign(ident=epi_clean.obs["leiden_scpoli_res3"].astype(str)).groupby(["ident","cell_type_pred"],observed=False).size().reset_index(name="n_cells")
def sk(x):
    try:return int(x)
    except:return x
summary_long["ident_sort"]=summary_long["ident"].map(sk)
summary_long=summary_long.sort_values(["ident_sort","n_cells"],ascending=[True,False]).drop(columns=["ident_sort"])
print(summary_long.to_string(index=False))
summary_long.to_csv(os.path.join(outdir,"cluster_celltype_summary.csv"),index=False)
dataset_col=next((c for c in ["dataset_id","dataset","sample"] if c in epi_clean.obs),None)
if dataset_col:
    tab=epi_clean.obs.loc[epi_clean.obs["leiden_scpoli_res3"].astype(str)=="42",dataset_col].value_counts()
    print(tab)
    tab.to_csv(os.path.join(outdir,f"cluster_42_{dataset_col}_composition.csv"))

In [ ]:
clean_path=os.path.join(outdir,f"epi_clean_{run_tag}.h5ad")
mid_path=os.path.join(outdir,f"epi_before_purity_filter_{run_tag}.h5ad")
epi_clean.write(clean_path)
epi.write(mid_path)
print("done")
print("cleaned:",clean_path)
print("before_purity:",mid_path)
print("outdir:",outdir)